In [1]:
import pandas as pd
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))
from helpers.data_loader import DataLoader

train = pd.read_parquet(DataLoader.transformed('train_t02.parquet'))
val   = pd.read_parquet(DataLoader.transformed('val_t02.parquet'))
test  = pd.read_parquet(DataLoader.transformed('test_t02.parquet'))
print('Train shape:', train.shape)
print('Val   shape:', val.shape)
print('Test  shape:', test.shape)
print()
print('Columns:', list(train.columns))

Train shape: (109693, 25)
Val   shape: (27483, 25)
Test  shape: (34294, 25)

Columns: ['Inspection ID', 'License #', 'Facility Type', 'Risk', 'Zip', 'Inspection Date', 'Inspection Type', 'Results', 'Violations', 'Latitude', 'Longitude', 'COMMUNITY AREA NAME', 'LICENSE DESCRIPTION', 'APPLICATION TYPE', 'LICENSE STATUS', 'violations_recorded', 'license_matched', 'has_prior_inspection', 'violation_count', 'inspection_year', 'inspection_month', 'inspection_dayofweek', 'inspection_quarter', 'days_to_license_expiry', 'license_expiry_missing']


In [2]:
train['_split'] = 'train'
val['_split']   = 'val'
test['_split']  = 'test'

combined = pd.concat([train, val, test]).sort_values(['License #', 'Inspection Date']).reset_index(drop=True)

def fail_rate_last_3(series):
    return series.shift(1).rolling(window=3, min_periods=1).mean()

combined['days_since_last_inspection'] = combined.groupby('License #')['Inspection Date'].diff().dt.days
combined['prev_inspection_result']     = combined.groupby('License #')['Results'].shift(1)
combined['fail_rate_last_3']           = combined.groupby('License #')['Results'].transform(fail_rate_last_3)

# Fill nulls using train stats only — no leakage into val or test
median_gap = combined.loc[combined['_split'] == 'train', 'days_since_last_inspection'].median()
prev_mean  = combined.loc[combined['_split'] == 'train', 'prev_inspection_result'].mean()
fail_mean  = combined.loc[combined['_split'] == 'train', 'fail_rate_last_3'].mean()

combined['days_since_last_inspection'] = combined['days_since_last_inspection'].fillna(median_gap)
combined['prev_inspection_result']     = combined['prev_inspection_result'].fillna(prev_mean)
combined['fail_rate_last_3']           = combined['fail_rate_last_3'].fillna(fail_mean)

train = combined[combined['_split'] == 'train'].drop(columns='_split').reset_index(drop=True)
val   = combined[combined['_split'] == 'val'].drop(columns='_split').reset_index(drop=True)
test  = combined[combined['_split'] == 'test'].drop(columns='_split').reset_index(drop=True)

print(f'median_gap={median_gap:.0f}  prev_mean={prev_mean:.4f}  fail_mean={fail_mean:.4f}')

median_gap=195  prev_mean=0.3496  fail_mean=0.3635


In [3]:
print('Inspection Type value counts:')
print(train['Inspection Type'].value_counts())

# Fit fail rate on train only
insp_fail_rate = train.groupby('Inspection Type')['Results'].mean()
global_mean = train['Results'].mean()

train['inspection_type_encoded'] = train['Inspection Type'].map(insp_fail_rate).fillna(global_mean)
val['inspection_type_encoded']   = val['Inspection Type'].map(insp_fail_rate).fillna(global_mean)
test['inspection_type_encoded']  = test['Inspection Type'].map(insp_fail_rate).fillna(global_mean)

print(f'\nNulls — train: {train["inspection_type_encoded"].isna().sum()}, '
      f'val: {val["inspection_type_encoded"].isna().sum()}, '
      f'test: {test["inspection_type_encoded"].isna().sum()}')

Inspection Type value counts:
Inspection Type
Canvass                              52094
License                              15483
Canvass Re-Inspection                11542
Complaint                            11029
License Re-Inspection                 6202
                                     ...  
fire complaint                           1
License consultation                     1
OWNER SUSPENDED OPERATION/LICENSE        1
CANVASS/SPECIAL EVENT                    1
Kids Cafe'                               1
Name: count, Length: 102, dtype: int64

Nulls — train: 0, val: 0, test: 0


In [4]:
# Fit fail rate on train only
facility_fail_rate = train.groupby('Facility Type')['Results'].mean()

train['facility_type_encoded'] = train['Facility Type'].map(facility_fail_rate).fillna(global_mean)
val['facility_type_encoded']   = val['Facility Type'].map(facility_fail_rate).fillna(global_mean)
test['facility_type_encoded']  = test['Facility Type'].map(facility_fail_rate).fillna(global_mean)

print(f'Nulls — train: {train["facility_type_encoded"].isna().sum()}, '
      f'val: {val["facility_type_encoded"].isna().sum()}, '
      f'test: {test["facility_type_encoded"].isna().sum()}')

Nulls — train: 0, val: 0, test: 0


In [5]:
train['is_revoked'] = (train['LICENSE STATUS'] == 'REV').astype(int)
val['is_revoked']   = (val['LICENSE STATUS']   == 'REV').astype(int)
test['is_revoked']  = (test['LICENSE STATUS']  == 'REV').astype(int)

In [6]:
APP_TYPE_MAP = {
    'ISSUE':  0,  # new license
    'RENEW':  1,  # renewal
    'C_LOC':  2,  # change of location
    'C_EXPA': 2,  # expansion
    'C_CAPA': 2,  # capacity change
    'C_SBA':  2,  # change of ownership
}

train['application_type_encoded'] = train['APPLICATION TYPE'].map(APP_TYPE_MAP).fillna(1)
val['application_type_encoded']   = val['APPLICATION TYPE'].map(APP_TYPE_MAP).fillna(1)
test['application_type_encoded']  = test['APPLICATION TYPE'].map(APP_TYPE_MAP).fillna(1)

In [7]:
NO_LONGER_NEEDED = [
    'Inspection ID', 'License #', 'Zip', 'Longitude', 'Latitude',
    'Violations', 'license_matched', 'Inspection Type', 'Facility Type',
    'COMMUNITY AREA NAME', 'LICENSE DESCRIPTION', 'LICENSE STATUS',
    'APPLICATION TYPE', 'Inspection Date', 'violations_recorded',
]

train.drop(columns=NO_LONGER_NEEDED, inplace=True)
val.drop(columns=NO_LONGER_NEEDED,   inplace=True)
test.drop(columns=NO_LONGER_NEEDED,  inplace=True)

print('\nRemaining columns:', train.columns.tolist())


Remaining columns: ['Risk', 'Results', 'has_prior_inspection', 'violation_count', 'inspection_year', 'inspection_month', 'inspection_dayofweek', 'inspection_quarter', 'days_to_license_expiry', 'license_expiry_missing', 'days_since_last_inspection', 'prev_inspection_result', 'fail_rate_last_3', 'inspection_type_encoded', 'facility_type_encoded', 'is_revoked', 'application_type_encoded']


In [8]:
train.to_parquet(DataLoader.transformed('train_t03.parquet'), index=False)
val.to_parquet(DataLoader.transformed('val_t03.parquet'),   index=False)
test.to_parquet(DataLoader.transformed('test_t03.parquet'),  index=False)
print('Saved train_t03.parquet, val_t03.parquet, and test_t03.parquet')

Saved train_t03.parquet, val_t03.parquet, and test_t03.parquet
